# Fine-Tuning GPT-2 Small with LoRA and GPU Acceleration

Welcome to the GPT-2 Production-Level Fine-Tuning workspace! This notebook guides you through running the PyTorch training pipeline from scratch on a Google Colab T4 GPU.

### Features
- Supervised Fine-Tuning (SFT) with Alpaca-style instruction dataset parsing.
- Parameter-Efficient LoRA (Low-Rank Adaptation) wrapping query and value projections.
- Hardware Optimizations: FP16 Automatic Mixed Precision (AMP) and Gradient Accumulation.
- Telemetry: Real-time training loss logging.

## 1. Setup and Environment Configuration

First, we check that we have a GPU active, switch working directories to the repository root, and install the required packages.

In [3]:
import os
import sys

# Walk up parent directories to find the project root containing training/train.py
found_root = False
for _ in range(4):
    if os.path.exists(os.path.join("training", "train.py")):
        found_root = True
        break
    os.chdir("..")

print("Working directory successfully resolved to:", os.getcwd())
if not found_root:
    print("WARNING: Could not locate repository root. Please ensure this notebook is run inside the repository.")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

Working directory successfully resolved to: c:\Users\Lenovo\Desktop\GPT-From-Scratch\GPT-PRODUCITON-LEVEL


In [4]:
# Verify GPU accessibility
!nvidia-smi

Sat Jun 13 10:26:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.49                 Driver Version: 596.49         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   69C    P0             16W /   85W |     410MiB /   4096MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
#Clone the repository (only needed if running on Google Colab from scratch)
# !git clone https://github.com/amoghsamadhiya779-afk/GPT-PRODUCTION-LEVEL.git
# %cd GPT-PRODUCTION-LEVEL

In [5]:
# Install dependencies (select appropriate command depending on platform)
# For Google Colab / Linux:
# !pip install -r requirements.txt
# For local Windows:
!py -m pip install -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Prepare the Dataset

You can upload your own custom data to Colab or use a default mock dataset. 
Let's write a sample `custom_instructions.json` file to demonstrate how the instruction parser handles Alpaca-style instruction datasets.

In [7]:
import json
import os

# Create a custom instruction dataset
custom_data = [
    {
        "instruction": "What is Python?",
        "input": "",
        "output": "Python is a high-level, interpreted programming language known for its readability, simplicity, and extensive ecosystem for machine learning and scientific computing."
    },
    {
        "instruction": "What is PyTorch?",
        "input": "",
        "output": "PyTorch is an open-source machine learning library developed by Meta's AI Research lab, widely used for deep learning applications such as natural language processing and computer vision."
    },
    {
        "instruction": "Explain the self-attention mechanism.",
        "input": "",
        "output": "Self-attention is a core mechanism in the Transformer architecture where input tokens are mapped into Query, Key, and Value vectors. Attention weights are computed using the scaled dot product of Queries and Keys, allowing tokens to weight context from all other tokens in parallel."
    }
]

# Ensure the data folder exists
os.makedirs("data", exist_ok=True)

with open("data/custom_instructions.json", "w", encoding="utf-8") as f:
    json.dump(custom_data, f, indent=2)

print("Saved custom instruction dataset to data/custom_instructions.json")

Saved custom instruction dataset to data/custom_instructions.json


## 3. Run the LoRA Finetuning Pipeline

We will now launch the training loop. We configure:
- `--lora`: Only train adapter parameters (extremely memory efficient).
- `--data_type instruction`: Enable target loss masking (masking prompt tokens with `-100`).
- `--use_amp`: Enable FP16 Mixed Precision for high-speed computation.
- `--accum_steps 2`: Accumulate gradients over 2 micro-batches before optimization step.

### Load Official Pre-trained OpenAI GPT-2 Weights
Since we are performing parameter-efficient fine-tuning using LoRA, the base model weights must be pre-trained to understand English. Otherwise, wrapping a randomly-initialized model with frozen parameters will produce gibberish.

Run this script to retrieve the official GPT-2 Small (124M) weights from Hugging Face and map them to our custom model architecture:

In [8]:
# Download and translate OpenAI GPT-2 weights
!python training/load_pretrained.py


  GPT-2 Pretrained Weights Loader

  [1/3] Downloading official GPT-2 Small (124M) weights from Hugging Face...
  [2/3] Mapping weights to custom model architecture...
  [3/3] Successfully saved mapped GPT-2 weights to: checkpoints\best_model.pt

  LOAD COMPLETE — Ready for text generation!




Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3402.84it/s]


In [9]:
# Run finetuning (using PyTorch AMP + LoRA + Pre-trained Checkpoint)
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data data/custom_instructions.json \
    --data_type instruction \
    --checkpoint checkpoints/best_model.pt \
    --lora \
    --lora_r 4 \
    --lora_alpha 8.0 \
    --accum_steps 2 \
    --use_amp

10:28:58 | INFO     | Using device: cpu
10:28:58 | INFO     | Loading instruction dataset from: data/custom_instructions.json
10:28:58 | INFO     | Train batches: 1 | Val batches: 1
10:28:58 | INFO     | Loading starting model weights from checkpoint: checkpoints/best_model.pt
Traceback (most recent call last):
  File "c:\Users\Lenovo\Desktop\GPT-From-Scratch\GPT-PRODUCITON-LEVEL\training\train.py", line 542, in <module>
    main()
  File "c:\Users\Lenovo\Desktop\GPT-From-Scratch\GPT-PRODUCITON-LEVEL\training\train.py", line 526, in main
    train(
  File "c:\Users\Lenovo\Desktop\GPT-From-Scratch\GPT-PRODUCITON-LEVEL\training\train.py", line 164, in train
    model_cfg = GPTConfig(**cfg_dict)
                ^^^^^^^^^^^^^^^^^^^^^
TypeError: GPTConfig.__init__() got an unexpected keyword argument 'drop_rate'


## 4. Verify Checkpoint and Inference Serving

Once training completes, the LoRA checkpoint is saved to `checkpoints/best_model.pt`.
We can instantiate the `GPTInferenceEngine` to verify that the model correctly loads the LoRA adapter and outputs the custom facts we trained it on.

In [31]:
import os
import sys
# Resolve project root directory if running from a subdirectory (like notebooks/)
for _ in range(4):
    if os.path.exists(os.path.join("training", "train.py")):
        break
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from app.inference import GPTInferenceEngine

# Load the newly trained LoRA checkpoint
engine = GPTInferenceEngine("checkpoints/best_model.pt")

# Generate text
res = engine.generate("What is PyTorch?", max_new_tokens=40, temperature=0.7, top_p=0.9, repetition_penalty=1.2)
print("\n--- Generated Text ---")
print(res["generated_text"])
print(f"Latency: {res['time_taken_seconds']:.3f}s | Speed: {res['tokens_per_second']:.1f} t/s")


--- Generated Text ---
What is PyTorch?Love Stellar conferred cancer Based after213 1895 indictment dash dwarf patriarch moderatelyerest Necerk resultediris muzzlelords confess mistake diminishing Ashes 1948�ibe worsenrama Spotlight heartbreaking roaming imaginedtailedTheoslov Cavecase patio condolences
Latency: 4.674s | Speed: 8.6 t/s


## 5. Deploying Your Checkpoint

To use this adapter in your local app or Hugging Face Space:
1. Download `checkpoints/best_model.pt` from the Colab file browser.
2. Copy the file into the `checkpoints/` folder of your project repository.
3. Re-launch the server! The server will automatically detect the checkpoint, inject the LoRA layers on the base GPT-2 model, and serve the adapter model.

## 6. Download Hugging Face Cosmopedia Math & Prompts Chat Dataset

We stream math textbooks from Hugging Face's Cosmopedia and merge them with Awesome ChatGPT Prompts templates (loaded using pandas) to compile our training corpus.

In [32]:
# Install datasets and pyarrow, then run the downloader
!python -m pip install datasets pyarrow pandas
!python data/download_cosmopedia.py


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



  Hugging Face Cosmopedia Math Dataset Downloader

Consolidating math textbooks to: data\cosmopedia_math.txt
Streaming samples from Hugging Face...
  Streaming 'auto_math_text' configuration...
    [OK] Loaded 2000 samples from 'auto_math_text'
  Streaming 'khanacademy' configuration...
    [OK] Loaded 2000 samples from 'khanacademy'
  Streaming 'openstax' configuration...
    [OK] Loaded 2000 samples from 'openstax'
Loading prompts from Hugging Face 'fka/prompts.chat' using pandas...
    [OK] Loaded 1875 samples from 'fka/prompts.chat'

  Success: Math corpus generated!
  Total samples merged : 7,875
  Saved location       : data\cosmopedia_math.txt
  Corpus file size     : 23.68 MB



Got disconnected from remote data host. Retrying in 5sec [1/20]
Exception ignored in: <function ResourceTracker.__del__ at 0x0000013E077162A0>
Traceback (most recent call last):
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\multiprocess\resource_tracker.py", line 80, in __del__
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\multiprocess\resource_tracker.py", line 89, in _stop
  File "c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\multiprocess\resource_tracker.py", line 102, in _stop_locked
AttributeError: '_thread.RLock' object has no attribute '_recursion_count'


## 7. Pre-Train Causal GPT-2 on Merged Math & Prompts Corpus

We pre-train our scratch-built GPT-2 model on the consolidated mathematical textbook and chatbot prompts corpus.

In [ ]:
# Run causal pre-training on the merged textbook and prompts corpus dataset
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data data/cosmopedia_math.txt \
    --epochs 3 \
    --batch_size 4 \
    --eval_freq 500 \
    --eval_iter 10 \
    --use_amp

### Alternative: Stream Directly from Hugging Face (No Download Needed)

If you want to train without writing the merged raw dataset to your local disk, you can stream the datasets directly from Hugging Face on-the-fly during training. This requires no local storage space.

In [ ]:
# Run causal pre-training by streaming Cosmopedia and prompts.chat on-the-fly
!python training/train.py \
    --config configs/gpt2_small.yaml \
    --data stream_hf \
    --data_type stream_hf \
    --epochs 3 \
    --batch_size 4 \
    --steps_per_epoch 1000 \
    --eval_freq 500 \
    --eval_iter 10 \
    --use_amp

## 8. Verify Pre-Trained Model Inference

Once pre-training completes, the model weights are saved in both `checkpoints/` and the central `models/` directory.
We can load the model checkpoint and verify its text generation capabilities using the `GPTInferenceEngine`.

In [ ]:
import os
import sys
# Resolve project root directory if running from a subdirectory (like notebooks/)
for _ in range(4):
    if os.path.exists(os.path.join("training", "train.py")):
        break
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from app.inference import GPTInferenceEngine

# Load the newly pre-trained model from the models folder
engine = GPTInferenceEngine("models/best_model.pt")

# Generate a math prompt response
res = engine.generate("Persona: Math Tutor\nPrompt: Solve x^2 = 9 for x.", max_new_tokens=50, temperature=0.7, top_p=0.9)
print("\n--- Generated Text ---")
print(res["generated_text"])
print(f"Latency: {res['time_taken_seconds']:.3f}s | Speed: {res['tokens_per_second']:.1f} t/s")

## 9. Model Storage Optimization & Hugging Face Upload

### Step 1: Prune Checkpoint for Efficient Storage & Speed
During pre-training/fine-tuning, checkpoints store the model weights along with the optimizer states (e.g. AdamW momentum states) to allow training resumption. This creates a file size of **1.95 GB**.

If you only need the model for **inference or serving** (running the FastAPI app, the Next.js frontend, or Streamlit), we can strip the optimizer states. This shrinks the file size to **~500 MB** (a 4x savings) while keeping it 100% functional.

In [ ]:
import torch
import os

checkpoint_path = "checkpoints/best_model.pt"

if os.path.exists(checkpoint_path):
    orig_size = os.path.getsize(checkpoint_path) / (1024 * 1024)
    print(f"Original Checkpoint Size: {orig_size:.2f} MB")
    
    # Load state dict and remove optimizer parameters
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    checkpoint.pop("optimizer_state_dict", None)
    
    # Save pruned version
    torch.save(checkpoint, checkpoint_path)
    
    new_size = os.path.getsize(checkpoint_path) / (1024 * 1024)
    print(f"Pruned Checkpoint Size  : {new_size:.2f} MB")
    print(f"Storage space saved     : {orig_size - new_size:.2f} MB!")
else:
    print(f"ERROR: Checkpoint not found at: {checkpoint_path}")

### Step 2: Upload to Hugging Face Spaces (Cloud-to-Cloud)
Instead of downloading the model to your local PC and then uploading it to Hugging Face, you can push it **directly from Colab to your Hugging Face Space** using the `huggingface_hub` Python API (pre-installed in Colab).

1. Generate a **Write Token** in your Hugging Face account under Settings -> Access Tokens.
2. Fill in your Space Repository ID (e.g. `your-username/your-space-name`) and access token below to upload.

In [ ]:
# Install Hugging Face Hub (usually pre-installed in Colab)
!python -m pip install huggingface_hub --quiet

from huggingface_hub import HfApi
import getpass

# Get configuration inputs interactively (securely hides your token and avoids public exposure)
repo_id = input("Enter your Hugging Face Space ID (e.g., username/space-name): ")
hf_token = getpass.getpass("Enter your Hugging Face WRITE access token: ")

api = HfApi()

print("Starting cloud-to-cloud upload to Hugging Face...")
try:
    api.upload_file(
        path_or_fileobj="checkpoints/best_model.pt",
        path_in_repo="checkpoints/best_model.pt",
        repo_id=repo_id,
        repo_type="space",
        token=hf_token,
    )
    print("\n=======================================================")
    print(" SUCCESS: Model uploaded directly to Hugging Face Space!")
    print("=======================================================")
except Exception as e:
    print(f"Upload failed: {e}")